# Kapitel 15: Anwendungen Entwickeln

> "Wissen ist nur dann wertvoll, wenn man es in die Praxis umsetzt."
> — **Anton Tschechow**, Schriftsteller

---

## Was Sie lernen werden

- Wie man eine einfache Chat-Anwendung mit Gesprächsverlauf erstellt
- Das RAG-Muster (Retrieval-Augmented Generation) zum Zugriff auf externes Wissen
- Wie man Dokumente in Chunks aufteilt, Einbettungen erstellt und relevante Informationen abruft
- Grundlegende Tool-Calling-Muster, die Modellen ermöglichen, Aktionen auszuführen
- Evaluierungsstrategien zum Testen, ob Ihre Anwendung tatsächlich funktioniert

---

## Setup

Zuerst installieren wir die erforderlichen Pakete und richten **Ollama** für kostenlose lokale LLM-Inferenz ein.

> **Warum Ollama?** Es ist völlig kostenlos, funktioniert offline und läuft auf jedem Computer.
> Keine API-Schlüssel oder Kreditkarten erforderlich. Viele Produktionsanwendungen verwenden jetzt lokale
> Modelle aus Datenschutz- und Kostengründen.

In [ ]:
# Erforderliche Pakete installieren
!pip install -q sentence-transformers numpy requests

# === OLLAMA SETUP ===
# Ollama läuft lokal - völlig kostenlos, kein API-Schlüssel erforderlich!

print("Ollama wird installiert...")
!curl -fsSL https://ollama.com/install.sh | sh

# Ollama-Server im Hintergrund starten
import subprocess
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import time
time.sleep(3)  # Auf Serverstart warten

# Ein kleines Modell herunterladen (~2GB Download, einmalig)
print("\nllama3.2-Modell wird heruntergeladen (kann beim ersten Mal einige Minuten dauern)...")
!ollama pull llama3.2

# Unsere Hilfsbibliothek herunterladen
!wget -q https://raw.githubusercontent.com/FirstLLM/code/main/llm_helper.py

print("\n✓ Setup abgeschlossen! Sie können jetzt kostenlos lokale LLMs verwenden.")

In [ ]:
# ===== IMPORTS =====
import os
import json
import re
from datetime import datetime

import numpy as np

# Unseren LLM-Helper importieren
from llm_helper import chat, chat_with_history
print("✓ llm_helper geladen")

# Prüfen, ob sentence-transformers verfügbar ist (für RAG-Einbettungen)
try:
    from sentence_transformers import SentenceTransformer
    print("✓ sentence-transformers installiert")
except ImportError:
    print("⚠ sentence-transformers nicht gefunden. Ausführen: pip install sentence-transformers")

In [ ]:
# ===== OLLAMA-VERBINDUNG TESTEN =====
# Überprüfen, ob Ollama läuft und das Modell verfügbar ist

print("Ollama-Verbindung wird getestet...")
try:
    response = chat("Sag 'Verbindung erfolgreich!' und nichts anderes.", temperature=0)
    print(f"✓ Ollama funktioniert!")
    print(f"  Antwort: {response}")
except Exception as e:
    print(f"⚠ Ollama-Verbindung fehlgeschlagen: {e}")
    print("  Stellen Sie sicher, dass Ollama läuft: ollama serve")

## 1. Eine Chat-Schleife erstellen

Beginnen wir mit einer einfachen Chat-Anwendung, die sich den Gesprächsverlauf merkt.

In [ ]:
class ChatSession:
    """Chat-Sitzung mit Verlaufsverwaltung."""
    
    def __init__(self, max_history=20):
        self.max_history = max_history
        self.history = []
        self.system_prompt = "Du bist ein hilfreicher Assistent."
    
    def count_messages(self):
        """Nachrichten im Gesprächsverlauf zählen."""
        return len(self.history)
    
    def trim_history_if_needed(self):
        """Älteste Nachrichten entfernen, wenn das Limit überschritten wird."""
        while len(self.history) > self.max_history:
            # Das älteste Benutzer-/Assistenten-Paar entfernen
            self.history.pop(0)
            if self.history and self.history[0]["role"] == "assistant":
                self.history.pop(0)
    
    def chat(self, user_message):
        """Eine Benutzernachricht verarbeiten und die Antwort zurückgeben."""
        # Benutzernachricht hinzufügen
        self.history.append({"role": "user", "content": user_message})
        
        # Bei Bedarf kürzen
        self.trim_history_if_needed()
        
        # Antwort mit unserem Helper abrufen
        assistant_message = chat_with_history(
            self.history,
            system=self.system_prompt
        )
        
        # Zum Verlauf hinzufügen
        self.history.append({"role": "assistant", "content": assistant_message})
        
        return assistant_message

print("ChatSession-Klasse definiert!")

In [ ]:
# Die Chat-Sitzung testen
session = ChatSession()

# Erste Nachricht
response1 = session.chat("Hallo! Was ist 2+2?")
print(f"Sie: Hallo! Was ist 2+2?")
print(f"Assistent: {response1}")
print(f"Nachrichten im Verlauf: {session.count_messages()}\n")

# Folgefrage (demonstriert das Gedächtnis)
response2 = session.chat("Was habe ich dich gerade gefragt?")
print(f"Sie: Was habe ich dich gerade gefragt?")
print(f"Assistent: {response2}")
print(f"Nachrichten im Verlauf: {session.count_messages()}")

## 2. RAG: Retrieval-Augmented Generation

RAG gibt Ihrem Modell Zugriff auf Dokumente, auf denen es nicht trainiert wurde.

Stellen Sie sich das wie eine **Prüfung mit offenen Unterlagen** vor: Das Modell kann Dinge nachschlagen, anstatt sich nur auf sein Gedächtnis zu verlassen.

In [ ]:
class SimpleRAG:
    """Ein einfaches RAG-System für Dokumentenabruf."""
    
    def __init__(self, embedding_model="all-MiniLM-L6-v2"):
        """Das RAG-System initialisieren."""
        self.encoder = SentenceTransformer(embedding_model)
        self.documents = []  # Originale Textteile (Chunks)
        self.embeddings = None  # NumPy-Array von Vektoren
        self.metadata = []  # Quellinformationen für Zitate
    
    def chunk_text(self, text, chunk_size=200, overlap=50):
        """Text in überlappende Chunks aufteilen."""
        words = text.split()
        chunks = []
        
        for i in range(0, len(words), chunk_size - overlap):
            chunk = " ".join(words[i:i + chunk_size])
            if chunk.strip():
                chunks.append(chunk)
        
        return chunks
    
    def add_document(self, text, source_name="unbekannt"):
        """Ein Dokument zur Wissensbasis hinzufügen."""
        chunks = self.chunk_text(text)
        
        for i, chunk in enumerate(chunks):
            self.documents.append(chunk)
            self.metadata.append({
                "source": source_name,
                "chunk_index": i
            })
        
        # Alle Dokumente neu einbetten
        self.embeddings = self.encoder.encode(
            self.documents,
            normalize_embeddings=True  # Wichtig für Kosinus-Ähnlichkeit
        )
        
        print(f"{len(chunks)} Chunks von '{source_name}' hinzugefügt")
    
    def retrieve(self, query, top_k=3, min_score=0.3):
        """Die relevantesten Chunks für eine Abfrage finden."""
        if self.embeddings is None or len(self.embeddings) == 0:
            return []
        
        # Die Abfrage einbetten
        query_embedding = self.encoder.encode(
            query,
            normalize_embeddings=True
        )
        
        # Ähnlichkeiten berechnen (Punktprodukt normalisierter Vektoren = Kosinus)
        similarities = np.dot(self.embeddings, query_embedding)
        
        # Top-k-Indizes ermitteln
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        
        # Ergebnisse erstellen, nach Mindestscore filtern
        results = []
        for idx in top_indices:
            score = float(similarities[idx])
            if score >= min_score:
                results.append({
                    "text": self.documents[idx],
                    "score": score,
                    "source": self.metadata[idx]["source"],
                    "index": int(idx)
                })
        
        return results

print("SimpleRAG-Klasse definiert!")

In [ ]:
# Beispieldokumente zum Testen
vacation_policy = """
Mitarbeiter erhalten 15 Tage bezahlten Urlaub pro Jahr.
Nicht genutzte Urlaubstage können ins nächste Jahr übertragen werden, maximal 5 Tage.
Urlaubsanträge müssen mindestens 2 Wochen im Voraus über das HR-Portal eingereicht werden.
Neue Mitarbeiter sind nach Abschluss ihrer 90-tägigen Probezeit urlaubsberechtigt.
"""

expense_policy = """
Geschäftsausgaben müssen innerhalb von 30 Tagen nach dem Ausgabendatum eingereicht werden.
Alle Ausgaben über 50 € erfordern einen Beleg. Mahlzeiten während Geschäftsreisen werden bis zu 75 € pro Tag erstattet.
Reichen Sie Spesenberichte über das Finanzportal mit entsprechenden Belegen ein.
Für Ausgaben über 500 € ist eine Genehmigung des Vorgesetzten erforderlich.
"""

remote_work_policy = """
Mitarbeiter können mit Genehmigung des Vorgesetzten bis zu 3 Tage pro Woche remote arbeiten.
Remote-Arbeiter müssen während der Kernarbeitszeit verfügbar sein: 10-15 Uhr in ihrer lokalen Zeitzone.
Home-Office-Ausstattung kann mit Genehmigung des Vorgesetzten bis zu 500 € erstattet werden.
Remote-Arbeitsverhältnisse sollten schriftlich dokumentiert werden.
"""

print("Beispieldokumente erstellt!")

In [ ]:
# RAG initialisieren und Dokumente hinzufügen
rag = SimpleRAG()

rag.add_document(vacation_policy, "urlaubsrichtlinie.txt")
rag.add_document(expense_policy, "spesenrichtlinie.txt")
rag.add_document(remote_work_policy, "remote-arbeitsrichtlinie.txt")

print(f"\nGesamtzahl der Dokumente in der Wissensbasis: {len(rag.documents)}")

In [ ]:
# Abruf testen
query = "Wie viele Urlaubstage bekomme ich?"
results = rag.retrieve(query)

print(f"Abfrage: '{query}'\n")
print("Abgerufene Dokumente:")
for i, doc in enumerate(results):
    print(f"\n[{i+1}] Score: {doc['score']:.3f} | Quelle: {doc['source']}")
    print(f"    {doc['text'][:100]}...")

In [ ]:
# Mit einer anderen Abfrage testen
query2 = "Was ist das Ausgabenlimit für Mahlzeiten?"
results2 = rag.retrieve(query2)

print(f"Abfrage: '{query2}'\n")
print("Abgerufene Dokumente:")
for i, doc in enumerate(results2):
    print(f"\n[{i+1}] Score: {doc['score']:.3f} | Quelle: {doc['source']}")
    print(f"    {doc['text'][:100]}...")

## 3. Prompting mit abgerufenem Kontext

Jetzt erstellen wir Prompts, die die abgerufenen Dokumente verwenden.

In [ ]:
def build_rag_prompt(query, retrieved_docs, min_score=0.3):
    """Einen Prompt mit abgerufenem Kontext und Zitationsanweisungen erstellen."""
    # Nach Score filtern
    good_docs = [d for d in retrieved_docs if d["score"] >= min_score]
    
    # Fall behandeln, wenn keine relevanten Dokumente gefunden wurden
    if not good_docs:
        return f"""Ich konnte keine relevanten Informationen finden, um Ihre Frage zu beantworten.

Frage: {query}

Bitte formulieren Sie Ihre Frage um oder lassen Sie mich wissen, ob ich auf Basis allgemeinen Wissens antworten soll."""
    
    # Kontext mit Zitationsmarkierungen erstellen
    context_parts = []
    for i, doc in enumerate(good_docs):
        source = doc.get("source", "unbekannt")
        context_parts.append(f"[{i+1}] (Quelle: {source})\n{doc['text']}")
    
    context = "\n\n".join(context_parts)
    
    return f"""Verwende die folgenden Quellen, um die Frage zu beantworten.
Zitiere Quellen mit [1], [2], etc. Verwende nur Informationen aus den bereitgestellten Quellen.
Wenn die Quellen die Antwort nicht enthalten, sage es.

Quellen:
{context}

Frage: {query}

Antwort:"""

print("build_rag_prompt() definiert!")

In [ ]:
# Sehen, wie der RAG-Prompt aussieht
query = "Wie viele Urlaubstage bekomme ich?"
docs = rag.retrieve(query)
prompt = build_rag_prompt(query, docs)

print("RAG-PROMPT:")
print("=" * 50)
print(prompt)

In [ ]:
def rag_answer(query, rag_system):
    """Eine Frage mit RAG beantworten."""
    # Relevante Dokumente abrufen
    docs = rag_system.retrieve(query, top_k=3)
    
    # Den Prompt erstellen
    prompt = build_rag_prompt(query, docs)
    
    # Antwort mit unserem Helper generieren
    response = chat(
        prompt,
        system="Du bist ein hilfreicher Assistent, der Fragen basierend auf bereitgestellten Quellen beantwortet. Zitiere immer deine Quellen.",
        temperature=0.3  # Niedrigere Temperatur für faktische Genauigkeit
    )
    
    return response

print("rag_answer() definiert!")

In [ ]:
# RAG-Beantwortung testen
questions = [
    "Wie viele Urlaubstage bekommen Mitarbeiter?",
    "Was ist das Ausgabenlimit für Mahlzeiten bei Geschäftsreisen?",
    "Kann ich von zu Hause aus arbeiten?",
]

for q in questions:
    print(f"F: {q}")
    answer = rag_answer(q, rag)
    print(f"A: {answer}\n")
    print("-" * 50 + "\n")

## 4. Tool-Calling-Grundlagen

Manchmal muss ein Modell mehr tun, als nur Informationen abzurufen. Es muss Aktionen ausführen.

In [ ]:
# Verfügbare Tools definieren
# WARNUNG: eval() wird hier der Einfachheit halber verwendet. In der Produktion verwenden Sie eine
# ordentliche Math-Parser-Bibliothek wie `simpleeval`, um Code-Injection-Angriffe zu verhindern.
TOOLS = {
    "calculate": {
        "description": "Grundlegende Arithmetik durchführen. Die Eingabe sollte ein mathematischer Ausdruck wie '2 + 2' oder '15 * 3' sein.",
        "function": lambda expr: str(eval(expr, {"__builtins__": {}}, {}))
    },
    "get_date": {
        "description": "Das aktuelle Datum abrufen.",
        "function": lambda: datetime.now().strftime("%Y-%m-%d")
    }
}

def parse_tool_call(response):
    """Toolaufruf aus der Modellausgabe extrahieren."""
    match = re.search(r'<tool>(\w+)\((.*)\)</tool>', response, re.DOTALL)
    if match:
        return match.group(1), match.group(2).strip()
    return None, None

def execute_tool(tool_name, argument):
    """Ein zugelassenes Tool sicher ausführen."""
    if tool_name not in TOOLS:
        return f"Fehler: Unbekanntes Tool '{tool_name}'"
    
    try:
        if argument:
            result = TOOLS[tool_name]["function"](argument)
        else:
            result = TOOLS[tool_name]["function"]()
        return str(result)
    except Exception as e:
        return f"Fehler bei der Ausführung von {tool_name}: {e}"

print("Tool-Funktionen definiert!")

In [ ]:
def chat_with_tools(user_message):
    """Chatten mit Tool-Calling-Funktion."""
    
    # Tool-Beschreibungen erstellen
    tool_descriptions = "\n".join(
        f"- {name}: {info['description']}"
        for name, info in TOOLS.items()
    )
    
    # Erster Durchgang: das Modell fragen
    prompt = f"""Du hast Zugriff auf diese Tools:
{tool_descriptions}

Um ein Tool zu verwenden, schreibe: <tool>name(argument)</tool>
Verwende ein Tool nur, wenn du es brauchst, um die Frage zu beantworten.

Benutzer: {user_message}
Assistent:"""
    
    first_response = chat(prompt, temperature=0)  # Deterministisch für zuverlässiges Parsing
    print(f"Erste Antwort des Modells: {first_response}")
    
    # Prüfen, ob das Modell ein Tool verwenden möchte
    tool_name, argument = parse_tool_call(first_response)
    
    if tool_name:
        # Das Tool ausführen
        tool_result = execute_tool(tool_name, argument)
        print(f"Tool ausgeführt: {tool_name}({argument}) = {tool_result}")
        
        # Zweiter Durchgang: Ergebnis zurück ans Modell geben
        followup = f"""{prompt}{first_response}

Tool-Ergebnis: {tool_result}

Gib jetzt deine endgültige Antwort an den Benutzer:"""
        
        final_response = chat(followup, temperature=0.3)
        return final_response
    
    # Kein Tool benötigt, erste Antwort zurückgeben
    return first_response

print("chat_with_tools() definiert!")

In [ ]:
# Tool-Calling testen
print("Tool-Calling wird getestet...\n")

# Berechnung testen
print("F: Was sind 15% von 847?")
answer = chat_with_tools("Was sind 15% von 847?")
print(f"Endgültige Antwort: {answer}\n")
print("-" * 50)

# Datum testen
print("\nF: Welches Datum haben wir heute?")
answer = chat_with_tools("Welches Datum haben wir heute?")
print(f"Endgültige Antwort: {answer}")

## 5. Evaluierung und Testen

Wie wissen Sie, ob Ihr RAG-System gut funktioniert? Erstellen Sie ein Evaluierungsset.

In [ ]:
def evaluate_response(response, expected_traits):
    """Eine Antwort anhand erwarteter Eigenschaften evaluieren."""
    results = {}
    response_lower = response.lower()
    
    # Auf Zitationsmarkierungen prüfen
    if "cites_source" in expected_traits:
        has_citation = bool(re.search(r'\[\d+\]', response))
        results["cites_source"] = (has_citation == expected_traits["cites_source"])
    
    # Auf erforderliche Schlüsselwörter prüfen
    if "contains_keywords" in expected_traits:
        keywords = expected_traits["contains_keywords"]
        all_present = all(kw.lower() in response_lower for kw in keywords)
        results["contains_keywords"] = all_present
    
    # Prüfen, ob es keine Ablehnung ist
    if "not_refusal" in expected_traits:
        refusal_phrases = ["ich weiß nicht", "ich kann nicht", "nicht in den quellen", "keine relevanten"]
        is_refusal = any(phrase in response_lower for phrase in refusal_phrases)
        if expected_traits["not_refusal"]:
            results["not_refusal"] = not is_refusal
        else:
            results["not_refusal"] = is_refusal
    
    # Minimale Länge prüfen
    if "min_words" in expected_traits:
        word_count = len(response.split())
        results["min_words"] = (word_count >= expected_traits["min_words"])
    
    return results

print("evaluate_response() definiert!")

In [ ]:
# Evaluierungsset definieren
eval_set = [
    {
        "query": "Wie viele Urlaubstage bekommen Mitarbeiter?",
        "expected_traits": {
            "contains_keywords": ["15", "tage"],
            "cites_source": True,
            "not_refusal": True
        }
    },
    {
        "query": "Was ist das Ausgabenlimit für Mahlzeiten?",
        "expected_traits": {
            "contains_keywords": ["75"],
            "cites_source": True,
            "not_refusal": True
        }
    },
    {
        "query": "Was ist der Sinn des Lebens?",  # Außerhalb des Bereichs!
        "expected_traits": {
            "not_refusal": False,  # Sollte ablehnen
            "cites_source": False
        }
    },
]

print(f"Evaluierungsset: {len(eval_set)} Testfälle")

In [ ]:
def run_evaluation(rag_system, eval_set):
    """Vollständige Evaluierung durchführen und Ergebnisse melden."""
    total_tests = 0
    passed_tests = 0
    
    for case in eval_set:
        print(f"\nAbfrage: {case['query']}")
        
        response = rag_answer(case["query"], rag_system)
        results = evaluate_response(response, case["expected_traits"])
        
        for trait, passed in results.items():
            total_tests += 1
            if passed:
                passed_tests += 1
                print(f"  [BESTANDEN] {trait}")
            else:
                print(f"  [FEHLGESCHLAGEN] {trait}")
        
        print(f"  Antwort: {response[:100]}...")
    
    print(f"\n{'='*50}")
    print(f"Ergebnisse: {passed_tests}/{total_tests} Tests bestanden ({100*passed_tests/total_tests:.1f}%)")

print("run_evaluation() definiert!")

In [ ]:
# Evaluierung ausführen
print("RAG-Evaluierung wird durchgeführt...")
run_evaluation(rag, eval_set)

## 6. Streaming-Antworten

Streaming lässt Antworten schneller erscheinen, indem die Ausgabe während der Generierung angezeigt wird.

In [ ]:
def stream_response(query, rag_system):
    """Eine RAG-Antwort mit Ollama streamen."""
    import requests
    
    docs = rag_system.retrieve(query, top_k=3)
    prompt = build_rag_prompt(query, docs)
    
    # Ollamas Streaming-Endpunkt direkt verwenden
    response = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": "llama3.2",
            "messages": [
                {"role": "system", "content": "Du bist ein hilfreicher Assistent."},
                {"role": "user", "content": prompt}
            ],
            "stream": True
        },
        stream=True
    )
    
    full_response = ""
    for line in response.iter_lines():
        if line:
            data = json.loads(line)
            if "message" in data and "content" in data["message"]:
                content = data["message"]["content"]
                print(content, end="", flush=True)
                full_response += content
    
    print()  # Zeilenumbruch am Ende
    return full_response

print("stream_response() definiert!")

In [ ]:
# Streaming testen
print("Streaming-Antwort wird getestet...\n")
print("F: Wie viele Tage kann ich remote arbeiten?\n")
print("A: ", end="")
stream_response("Wie viele Tage kann ich remote arbeiten?", rag)

## Übungen

### Übung 1: Eigene Wissensbasis erstellen

Erstellen Sie eine Wissensbasis zu einem Thema, das Ihnen wichtig ist.

In [ ]:
# IHR CODE HIER
# 1. Schreiben Sie 10 kurze Dokumente zu einem Thema (Rezepte, Spielregeln, Notizen)
# 2. Initialisieren Sie eine SimpleRAG-Instanz
# 3. Fügen Sie Ihre Dokumente hinzu
# 4. Testen Sie mit 5 Fragen
# 5. Beobachten Sie: Ruft es die richtigen Chunks ab?

### Übung 2: Temperatur-Experiment

Testen Sie, wie die Temperatur RAG-Antworten beeinflusst.

In [ ]:
# IHR CODE HIER
# 1. Modifizieren Sie rag_answer(), um einen Temperaturparameter zu akzeptieren
# 2. Stellen Sie dieselbe Frage mit den Temperaturen 0.3, 0.7, 1.0
# 3. Vergleichen Sie die Antworten
# 4. Welche ist am zuverlässigsten für faktische Fragen?

### Übung 3: Evaluierungsset erstellen

Erstellen Sie ein umfassendes Evaluierungsset für Ihre Wissensbasis.

In [ ]:
# IHR CODE HIER
# 1. Erstellen Sie 10 Evaluierungsfragen für Ihre Wissensbasis
# 2. Einbeziehen: 5 einfache, 3 schwierige, 2 adversariale (außerhalb des Bereichs)
# 3. Definieren Sie erwartete Eigenschaften für jede
# 4. Führen Sie die Evaluierung durch und melden Sie die Genauigkeit

### Übung 4: Checkpoint - Persönlicher Wissensassistent

Erstellen Sie eine vollständige "Mit Ihren Dokumenten chatten"-Anwendung.

In [ ]:
# IHR CODE HIER
# 1. Sammeln Sie 5-10 Textdateien (Notizen, Artikel, Dokumentation)
# 2. Laden Sie sie in eine SimpleRAG-Instanz
# 3. Erstellen Sie eine Chat-Schleife, die RAG für jede Antwort verwendet
# 4. Einbeziehen: Token-Zählung, Verlaufsverwaltung, Zitate
# 5. Erstellen Sie ein Evaluierungsset mit 10 Fragen
# 6. Führen Sie die Evaluierung durch und melden Sie die Genauigkeit

## Zusammenfassung

**Was wir gebaut haben:**

- Eine Chat-Schleife mit Verlaufsverwaltung und Token-Budget
- Ein vollständiges RAG-System von der Dokumentaufteilung bis zur Zitatgenerierung
- Eine Vorschau auf Tool-Calling mit Sicherheitsaspekten
- Evaluierung und Validierung für Produktionsbereitschaft

**Was wir gelernt haben:**

- Ihr Trainingswissen (Einbettungen, Chunking, Datenqualität, Reproduzierbarkeit) überträgt sich direkt auf Anwendungen
- RAG löst das Problem "Das Modell kennt meine Daten nicht"
- Tool-Calling erweitert die Möglichkeiten von Modellen, erfordert aber sorgfältige Sicherheitsmaßnahmen
- Evaluierung ist unverzichtbar, nicht optional